In [13]:
# Needed if running on Colab, comment out if in local environment!
!pip3 install open-spiel
!pip3 install torch
!pip3 install matplotlib
!pip3 install tqdm


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [14]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from go_search_problem import GoProblem, GoState
from heuristic_go_problems import GoProblemLearnedHeuristic, GoProblemSimpleHeuristic
from agents import GreedyAgent, RandomAgent, AlphaBetaAgent
import matplotlib.pyplot as plt
from tqdm import tqdm
from game_runner import GameRunner
import pickle

torch.set_default_tensor_type(torch.FloatTensor)

In [15]:
def load_dataset(path: str):
    with open(path, 'rb') as f:
        dataset = pickle.load(f)
    return dataset

# We've provided a dataset with pyspiel and without (i.e., pygo)
# dataset_5x5 = load_dataset('dataset_5x5.pkl')
dataset_5x5 = load_dataset('dataset_5x5_pygo.pkl')

In [16]:
def save_model(path: str, model):
    """
    Save model to a file
    Input:
        path: path to save model to
        model: Pytorch model to save
    """
    torch.save({
        'model_state_dict': model.state_dict(),
    }, path)

def load_model(path: str, model):
    """
    Load model from file

    Note: you still need to provide a model (with the same architecture as the saved model))

    Input:
        path: path to load model from
        model: Pytorch model to load
    Output:
        model: Pytorch model loaded from file
    """
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    return model

# Task 1: Convert GameState to Features

In [62]:
def get_features(game_state: GoState):
    """
    Map a game state to a list of features.

    Some useful functions from game_state include:
        game_state.size: size of the board
        get_pieces_coordinates(player_index): get coordinates of all pieces of a player (0 or 1)
        get_pieces_array(player_index): get a 2D array of pieces of a player (0 or 1)
        
        get_board(): get a 2D array of the board with 4 channels (player 0, player 1, empty, and player to move). 4 channels means the array will be of size 4 x n x n
    
        Descriptions of these methods can be found in the GoState

    Input:
        game_state: GoState to encode into a fixed size list of features
    Output:
        features: list of features
    """

     # Handle dict input from dataset
    if isinstance(game_state, dict):
        board = game_state['board']
        board_size = game_state['size']
        current_player = game_state['player_to_move']
        opponent_player = 1 - current_player
        is_terminal = game_state['is_terminal']
        terminal_val = 0  
        current_player_pieces = board[current_player]
        opponent_player_pieces = board[opponent_player]
        empty_spaces = board[2]
        current_player_coordinates = game_state['black_pieces'] if current_player == 0 else game_state['white_pieces']
        opponent_player_coordinates = game_state['white_pieces'] if current_player == 0 else game_state['black_pieces']

    # Handle GoState input during actual gameplay
    else:
        board_size = game_state.size

         # Return early for terminal states
        if game_state.is_terminal():
            terminal_val = game_state.terminal_value()[0] 
            return [terminal_val] + [0] * 10
        
        current_player = game_state.player_to_move()
        opponent_player = 1 - current_player
        terminal_val = game_state.terminal_value()[current_player] if game_state.is_terminal() else 0
        current_player_pieces = game_state.get_pieces_array(current_player)
        opponent_player_pieces = game_state.get_pieces_array(opponent_player)
        empty_spaces = game_state.get_empty_spaces()
        current_player_coordinates = game_state.get_pieces_coordinates(current_player)
        opponent_player_coordinates = game_state.get_pieces_coordinates(opponent_player)

    # Stone difference
    current_stones = current_player_pieces.sum()
    opponent_stones = opponent_player_pieces.sum()
    stone_difference = current_stones - opponent_stones

    # Liberty counts
    current_player_liberties = 0
    opponent_player_liberties = 0

    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    for coord in current_player_coordinates:
        row, col = coord
        for dr, dc in directions:
            nr, nc = row + dr, col + dc
            if 0 <= nr < board_size and 0 <= nc < board_size:
                if empty_spaces[nr, nc]:
                    current_player_liberties += 1
    
    for coord in opponent_player_coordinates:
        row, col = coord
        for dr, dc in directions:
            nr, nc = row + dr, col + dc
            if 0 <= nr < board_size and 0 <= nc < board_size:
                if empty_spaces[nr, nc]:
                    opponent_player_liberties += 1
                
    liberty_difference = current_player_liberties - opponent_player_liberties

    # Board coverage
    current_player_coverage = current_stones / (board_size * board_size)
    opponent_player_coverage = opponent_stones / (board_size * board_size)
    total_board_coverage = current_player_coverage + opponent_player_coverage

    features = []
    features.extend([  
        terminal_val,           # is game over and who won 
        current_player,         # whose turn it is
        stone_difference,       # current - opponent stones
        current_stones,         # raw stone count
        opponent_stones,        # raw stone count
        current_player_liberties,   # freedom of your stones
        opponent_player_liberties,  # freedom of opponent stones
        liberty_difference,     # liberty difference
        current_player_coverage,   # how much of the board you control
        opponent_player_coverage,  # how much of the board opponent controls
        total_board_coverage      # how much of the board is occupied
    ])

    

    return features

In [18]:
# Print information about first data point
data_point = dataset_5x5[0]
features = get_features(data_point[0])
action = data_point[1]
result = data_point[2]
print(data_point[0])
print("features", features)
print("Action #", action)
print("Game Result", result)

{'board': array([[[0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 1.],
        [0., 0., 1., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.]],

       [[1., 1., 0., 1., 1.],
        [1., 1., 0., 1., 1.],
        [1., 1., 0., 0., 1.],
        [1., 1., 1., 1., 0.],
        [1., 0., 0., 1., 1.]],

       [[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]]]), 'size': 5, 'player_to_move': 1, 'is_terminal': False, 'legal_actions': [0, 1, 3, 4, 5, 6, 8, 9, 10, 11, 14, 15, 16, 17, 18, 20, 23, 24, 25], 'black_pieces': array([[0, 2],
       [2, 2],
       [3, 4],
       [4, 2]]), 'white_pieces': array([[1, 2],
       [2, 3],
       [4, 1]])}
features [0, 1, np.float64(-1.0), np.float64(3.0), np.float64(4.0), 7, 9, -2, np.float64(0.12), np.

# Task 2: Supervised Learning of a Value Network

In [44]:
class ValueNetwork(nn.Module):
    def __init__(self, input_size):
      super(ValueNetwork, self).__init__()

      output_size = 1

      self.layer1 = nn.Linear(input_size, 32)
      self.activation1 = nn.ReLU()
        
      self.layer2 = nn.Linear(32, 16)
      self.activation2 = nn.ReLU()
        
      self.layer3 = nn.Linear(16, output_size)
      self.activation3 = nn.Tanh()

    def forward(self, x):
      """
      Run forward pass of network

      Input:
        x: input to network
      Output:
        output of network
      """
      x = self.layer1(x)
      x = self.activation1(x)
      x = self.layer2(x)
      x = self.activation2(x)
      x = self.layer3(x)
      x = self.activation3(x)
      
      return x

In [40]:
# This will not produce meaningful outputs until trained, but you can test for syntax errors
features_tensor = torch.Tensor(features)
value_net = ValueNetwork(len(features))
print("predicted Value", value_net(features_tensor))

predicted Value tensor([-0.3337], grad_fn=<TanhBackward0>)


In [60]:
def train_value_network(dataset, num_epochs, learning_rate):
    """
    Train a value network on the provided dataset.

    Input:
        dataset: list of (state, action, result) tuples
        num_epochs: number of epochs to train for
        learning_rate: learning rate for gradient descent
    Output:
        model: trained model
    """
    # Make sure dataset is shuffled for better performance
    random.shuffle(dataset)

    input_size = len(get_features(dataset[0][0]))
    model = ValueNetwork(input_size)

    loss_function = nn.MSELoss()

    # You can use Adam, which is stochastic gradient descent with ADAptive Momentum
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    batch_size = 32

    for epoch in range(num_epochs):
        batch_loss = 0
        batch_counter = 0
        for data_point in dataset:
            state = data_point[0]
            features = get_features(state)
            features_tensor = torch.tensor(features, dtype=torch.float32)

            # Note: You will have to convert the label to a torch tensor to use with torch's loss functions
            label = data_point[2]  # Win = 1, Loss = -1
            tensor_label = torch.tensor([label], dtype=torch.float32)

            prediction = model(features_tensor)

            loss = loss_function(prediction, tensor_label)
            batch_loss += loss
            batch_counter += 1
            if batch_counter % batch_size == 0:
                # Call backward to run backward pass and compute gradients
                batch_loss.backward()

                # Run gradient descent step with optimizer
                optimizer.step()

                # Reset gradient for next batch
                optimizer.zero_grad()
                batch_loss = 0

    return model

value_model = train_value_network(dataset_5x5, 100, 1e-4)
save_model("value_model.pt", value_model)

## Comparing Learned Value function against other Agents

In [64]:
class GoProblemLearnedHeuristic(GoProblem):
    def __init__(self, model=None, state=None):
        super().__init__(state=state)
        self.model = model

    def encoding(self, state):
        """
        Get encoding of state (convert state to features)
        Note, this may call get_features() from Task 1. 

        Input:
            state: GoState to encode into a fixed size list of features
        Output:
            features: list of features
        """
        features = get_features(state)

        return features

    def heuristic(self, state, player_index):
        """
        Return heuristic (value) of current state

        Input:
            state: GoState to encode into a fixed size list of features
            player_index: index of player to evaluate heuristic for
        Output:
            value: heuristic (value) of current state
        """
        features = self.encoding(state)
        features_tensor = torch.tensor(features, dtype=torch.float32)
        value = self.model(features_tensor)

        # Note, your agent may perform better if you force it not to pass
        # (i.e., don't select action #25 on a 5x5 board unless necessary)
        return value

    def __str__(self) -> str:
        return "Learned Heuristic"


def create_value_agent_from_model():
    """
    Create agent object from saved model. This (or other methods like this) will be how your agents will be created in gradescope and in the final tournament.
    """

    model_path = "value_model.pt"
    feature_size = 11
    model = load_model(model_path, ValueNetwork(feature_size))
    heuristic_search_problem = GoProblemLearnedHeuristic(model)

    learned_agent = GreedyAgent(heuristic_search_problem)

    return learned_agent

learned_agent = create_value_agent_from_model()
agent2 = GreedyAgent(GoProblemSimpleHeuristic())
print("Greedy Agent", agent2)
print("Learned Agent", learned_agent)

game_runner = GameRunner()
game_runner.play_tournament(learned_agent, agent2, num_games=100)

Greedy Agent GreedyAgent + Simple Heuristic
Learned Agent GreedyAgent + Learned Heuristic


Playing tournament: 100%|██████████| 50/50 [00:02<00:00, 21.10it/s]

Tournament Results
Games played: 100
GreedyAgent + Learned Heuristic wins: 54 (54.0%)
GreedyAgent + Simple Heuristic wins: 46 (46.0%)
GreedyAgent + Learned Heuristic wins as BLACK: 35
GreedyAgent + Simple Heuristic wins as BLACK: 31
GreedyAgent + Learned Heuristic avg move time: 0.001s
GreedyAgent + Simple Heuristic avg move time: 0.000s
GreedyAgent + Learned Heuristic min time remaining: 21.0s
GreedyAgent + Simple Heuristic min time remaining: 20.0s


TournamentStats(player1_wins=54, player2_wins=46, player1_wins_as_black=35, player2_wins_as_black=31, player1_total_time=np.float64(0.08922798633950535), player2_total_time=np.float64(0.016694623003793613), player1_min_time_remaining=20.991560697555542, player2_min_time_remaining=19.9985990524292, player1_max_move_time=np.float64(0.0026776790618896484), player2_max_move_time=np.float64(0.0003829002380371094), games_played=100)